In [ ]:
# Computer Vision Course (CSE 40535/60535)
# University of Notre Dame, Fall 2024
# ________________________________________________________________
# Adam Czajka, Jin Huang, September 2017 - 2024

import cv2
import numpy as np
from skimage import measure
import matplotlib.pyplot as plt

### Read our input image ###

In [ ]:
# Read the image
sample = cv2.imread('breakfast.png')

### Convert our image to grayscale and HSV images ###

In [ ]:
# Convert the original image to grayscale
sample_grey = cv2.cvtColor(sample, cv2.COLOR_BGR2GRAY)

# Convert the original image to HSV and take H channel
sample_hsv = cv2.cvtColor(sample, cv2.COLOR_BGR2HSV)
sample_h   = sample_hsv[:, :, 0]

# Show H channel of image (Feature 2 visualization)
plt.figure(figsize=(10, 6))
plt.imshow(sample_h, cmap='hsv')
plt.colorbar(label='Hue value')
plt.title('Feature 2: H channel of the image (Hue)')
plt.axis('off')
plt.tight_layout()
plt.savefig('feature2_hue_channel.png', dpi=150, bbox_inches='tight')
plt.show()

### Pre-process your images ###

In [ ]:
# Binarize the image using Otsu's method
ret1, binary_image = cv2.threshold(src=sample_grey, thresh=0, maxval=255,
                                   type=cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# *** TASK ***
# Fill holes inside donuts using flood-fill from corner (0,0).
# This converts ring-shaped blobs into solid filled circles so that
# each donut is one connected component instead of a ring + inner hole.
im_floodfill = binary_image.copy()
h, w = binary_image.shape[:2]
mask = np.zeros((h + 2, w + 2), np.uint8)
cv2.floodFill(im_floodfill, mask, (0, 0), 255)
im_floodfill_inv = cv2.bitwise_not(im_floodfill)
binary_image     = binary_image | im_floodfill_inv

# Show binary image (Feature 1 visualization — perimeter is measured on this)
plt.figure(figsize=(10, 6))
plt.imshow(binary_image, cmap='gray')
plt.title('Feature 1 source: binary image (after Otsu + flood-fill)')
plt.axis('off')
plt.tight_layout()
plt.savefig('feature1_binary_image.png', dpi=150, bbox_inches='tight')
plt.show()

### Find the objects and extract their features ###

In [ ]:
# Find connected pixels and compose them into objects
labels     = measure.label(binary_image)
properties = measure.regionprops(labels)

# Filter out very small regions (noise) and the image background
properties = [p for p in properties if 200 < p.area < 60000]

# *** TASK ***
# Calculate two features for each detected object:
#   Feature 1 (dimension 0): perimeter — geometry-based.
#              Squares have jagged waffle edges → larger perimeter (~375–445).
#              Donuts are smooth circles → smaller perimeter (~225–325).
#   Feature 2 (dimension 1): mean value of the H (hue) channel inside the
#              object's bounding box — color-based.
#              Blue Fruit Loops have high hue (~57–70).
#              Red Fruit Loops and Chex squares have low hue (~9–24).

features = np.zeros((len(properties), 2))
for i, prop in enumerate(properties):
    # Feature 1: perimeter of the region
    features[i, 0] = prop.perimeter

    # Feature 2: mean hue inside bounding box
    min_row, min_col, max_row, max_col = prop.bbox
    single_object_hue = sample_h[min_row:max_row, min_col:max_col]
    features[i, 1]    = np.mean(single_object_hue)

### We have features. Time to classify! ###

In [ ]:
# *** TASK ***
# Show our objects in the 2D feature space.
# The two red lines mark the classification thresholds.

plt.figure(figsize=(8, 6))
plt.plot(features[:, 0], features[:, 1], 'ro', markersize=8)
plt.xlabel('Feature 1: perimeter')
plt.ylabel('Feature 2: average grayscale value of H channel')
plt.title('Detected objects in our 2D feature space (Task 1)')

# *** TASK ***
# Choose classification thresholds by visual inspection of the scatter plot.
#   thrF1 separates squares (perimeter > thrF1) from donuts (perimeter <= thrF1).
#   thrF2 separates blue donuts (hue > thrF2) from red donuts (hue <= thrF2).
thrF1 = 335   # perimeter threshold
thrF2 = 45    # hue threshold

plt.axvline(x=thrF1, color='r', linewidth=1.5, label=f'thrF1={thrF1}')
plt.axhline(y=thrF2, color='r', linewidth=1.5, label=f'thrF2={thrF2}')
plt.legend()
plt.tight_layout()
plt.savefig('task1_feature_space.png', dpi=150, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────
# Count and display the objects
squares      = 0
blue_circles = 0
red_circles  = 0

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))

# *** TASK ***
# Classification rules (2D threshold classifier):
#   squares     : perimeter > thrF1
#   blue donuts : perimeter <= thrF1  AND  hue > thrF2
#   red donuts  : perimeter <= thrF1  AND  hue <= thrF2
for i in range(len(properties)):
    cx = np.round(properties[i].centroid[1])
    cy = np.round(properties[i].centroid[0])

    if features[i, 0] > thrF1:
        squares += 1
        ax.plot(cx, cy, '.g', markersize=15)

    elif features[i, 0] <= thrF1 and features[i, 1] > thrF2:
        blue_circles += 1
        ax.plot(cx, cy, '.b', markersize=15)

    elif features[i, 0] <= thrF1 and features[i, 1] <= thrF2:
        red_circles += 1
        ax.plot(cx, cy, '.r', markersize=15)

ax.set_title('Task 1 result: green=square, blue=blue donut, red=red donut')
ax.axis('off')
plt.tight_layout()
plt.savefig('task1_result.png', dpi=150, bbox_inches='tight')
plt.show()

print("I found %d squares, %d blue donuts, and %d red donuts." %
      (squares, blue_circles, red_circles))

### Task 2: replace the least discriminating feature with a Gabor texture feature ###

**Observation from Task 1:**
- *Feature 1 (perimeter)* cleanly separates squares from donuts — highly discriminating.
- *Feature 2 (hue)* separates blue donuts from the rest, but squares and red donuts share
  similar hue (~9–24), making hue the **least discriminating** feature overall.

**Task 2 strategy:**
Replace Feature 1 (perimeter) with the **Gabor texture energy** computed locally inside
each detected object's region.
Chex squares have a strong waffle/grid texture → high Gabor energy.
Fruit Loop donuts are smooth rings → low Gabor energy.
Feature 2 (hue) is kept unchanged to distinguish blue from red donuts.

**Relationship to Homework 4:**
HW4 applied a single Gabor kernel (`cv2.getGaborKernel` + `cv2.filter2D`) at orientation
θ=0 to find vertical stripes. Task 2 re-uses those same two building blocks and extends them:

1. **4 orientations** (0°, 45°, 90°, 135°) instead of just θ=0 → rotation invariance
2. **Quadrature pair** (real + imaginary phase kernels) → phase invariance (Gabor *energy*)
   instead of a single-phase response
3. **Local extraction** — average energy inside the object's segmentation mask, applied
   to the locations where objects were found (as suggested in the instructions)

```python
# HW4 pattern (single orientation, single phase):
kernel = cv2.getGaborKernel((ksize,ksize), sigma, theta=0, lbd, gamma, psi=0, cv2.CV_32F)
res    = cv2.filter2D(gray, cv2.CV_8UC3, kernel)

# Task 2 extension (4 orientations, quadrature pair → energy):
for theta in [0, pi/4, pi/2, 3pi/4]:
    kern_r = cv2.getGaborKernel(..., psi=0,    ...)   # real part
    kern_i = cv2.getGaborKernel(..., psi=pi/2, ...)   # imaginary part
    energy += sqrt(filter2D(kern_r)^2 + filter2D(kern_i)^2)
```

In [ ]:
# ── Task 2 ──────────────────────────────────────────────────────────────────
# Re-uses the same binary segmentation (labels / properties) from Task 1.
#
# Gabor texture energy — extends Homework 4's approach:
#   HW4: cv2.getGaborKernel + cv2.filter2D at a single orientation (θ=0)
#   Here: same two building blocks, generalized to:
#     - 4 orientations for rotation invariance
#     - quadrature pair (psi=0 real + psi=π/2 imaginary) for phase invariance
#     - local extraction: average energy within each object's segmentation mask

sample_grey_f32 = sample_grey.astype(np.float32) / 255.0

# Gabor parameters — same sigma and gamma as HW4; lambda tuned to the
# Chex waffle grid spacing (~25–35 px in the image)
ksize = 21
sigma = 4.0    # HW4 used 4.0
lbd   = 30.0   # HW4 used 4.0 for fine vertical stripes; 30.0 matches the coarser Chex grid
gamma = 0.5    # spatial aspect ratio
psi_r = 0.0    # phase = 0   → real (cosine) part of Gabor
psi_i = np.pi / 2  # phase = π/2 → imaginary (sine) part of Gabor

# Build Gabor energy map: sum quadrature-pair energy across 4 orientations
# (directly re-using cv2.getGaborKernel and cv2.filter2D from HW4)
gabor_map = np.zeros_like(sample_grey_f32)
for theta in [0, np.pi/4, np.pi/2, 3*np.pi/4]:
    kern_real = cv2.getGaborKernel((ksize, ksize), sigma, theta, lbd, gamma, psi_r, cv2.CV_32F)
    kern_imag = cv2.getGaborKernel((ksize, ksize), sigma, theta, lbd, gamma, psi_i, cv2.CV_32F)
    resp_r    = cv2.filter2D(sample_grey_f32, cv2.CV_32F, kern_real)
    resp_i    = cv2.filter2D(sample_grey_f32, cv2.CV_32F, kern_imag)
    gabor_map += np.sqrt(resp_r**2 + resp_i**2)   # Gabor energy per orientation

# Visualise the Gabor energy map (bright = strongly textured = squares)
plt.figure(figsize=(10, 6))
plt.imshow(gabor_map, cmap='hot')
plt.colorbar(label='Gabor energy')
plt.title('Task 2 — Feature 1: Gabor energy map (λ=30, 4 orientations, quadrature pair)')
plt.axis('off')
plt.tight_layout()
plt.savefig('task2_gabor_map.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Extract features for Task 2 ─────────────────────────────────────────────
features2 = np.zeros((len(properties), 2))
for i, prop in enumerate(properties):
    r0, c0, r1, c1 = prop.bbox

    # Feature 1 (Task 2): average Gabor energy inside the object's segmentation mask
    # (applied locally "in places where an object was found" — per the instructions)
    obj_mask        = (labels == prop.label)[r0:r1, c0:c1]
    patch_gabor     = gabor_map[r0:r1, c0:c1]
    features2[i, 0] = np.mean(patch_gabor[obj_mask > 0])

    # Feature 2 (Task 2): same hue feature as Task 1
    features2[i, 1] = np.mean(sample_h[r0:r1, c0:c1])

# ── Classification thresholds ────────────────────────────────────────────────
thrG = 357.0   # Gabor energy: squares > thrG (~370–410), donuts <= thrG (~300–350)
thrH = 45      # hue threshold (same as Task 1)

# ── Feature space plot ───────────────────────────────────────────────────────
plt.figure(figsize=(8, 6))
plt.plot(features2[:, 0], features2[:, 1], 'ro', markersize=8)
plt.axvline(x=thrG, color='r', linewidth=1.5, label=f'thrG={thrG}')
plt.axhline(y=thrH, color='r', linewidth=1.5, label=f'thrH={thrH}')
plt.xlabel('Feature 1: avg Gabor energy (λ=30, quadrature, within object mask)')
plt.ylabel('Feature 2: average grayscale value of H channel')
plt.title('Detected objects in 2D feature space (Task 2)')
plt.legend()
plt.tight_layout()
plt.savefig('task2_feature_space.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Classify and annotate result ─────────────────────────────────────────────
squares2      = 0
blue_circles2 = 0
red_circles2  = 0

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))

for i in range(len(properties)):
    cx = np.round(properties[i].centroid[1])
    cy = np.round(properties[i].centroid[0])

    if features2[i, 0] > thrG:
        squares2 += 1
        ax.plot(cx, cy, '.g', markersize=15)

    elif features2[i, 0] <= thrG and features2[i, 1] > thrH:
        blue_circles2 += 1
        ax.plot(cx, cy, '.b', markersize=15)

    elif features2[i, 0] <= thrG and features2[i, 1] <= thrH:
        red_circles2 += 1
        ax.plot(cx, cy, '.r', markersize=15)

ax.set_title('Task 2 result (Gabor energy + hue): green=square, blue=blue donut, red=red donut')
ax.axis('off')
plt.tight_layout()
plt.savefig('task2_result.png', dpi=150, bbox_inches='tight')
plt.show()

print("Task 2: I found %d squares, %d blue donuts, and %d red donuts." %
      (squares2, blue_circles2, red_circles2))